# `AdvectionParams` — Pipeline & Boundary-Condition Workflow

This notebook documents the five-stage initialisation and solving pipeline for `AdvectionParams`.
It shows what each stage reads, what it writes, and how the two alternative BC methods
(`update_surface_BCs_from_reference` vs `update_surface_BCs_from_zom`) and the two
reconciliation modes (`update_ustar` vs `update_H`) differ.

---

## Stage overview

| # | Stage | Key method(s) | Writes |
|---|-------|---------------|--------|
| ① | Construction | `__post_init__` | `Q_c`, `Q_f`, `Q_a` (canonical moisture) |
| ② | Radiation closure | `solve_surface_radiation_inplace(fix=…)` | `SW_in`, free albedo, `Rn_*`, `LW_down` |
| ③ | Surface BCs | `update_surface_BCs_from_reference()` **or** `update_surface_BCs_from_zom()` | `T_sc`, `T_sf`, `Q_c`, `Q_f` |
| ④ | Heat reconciliation | `reconcile_heat_with_prescribed_surface_T(mode=…, ref=…)` | `ustar_f/c` **or** `H_f + le_factor` |
| ⑤ | Advection solver | `uniform_T/Q` → `integrate_T/H2O_step` (Thomas) | `T(x,z)`, `Q(x,z)`, flux arrays |

---

## Full pipeline diagram (Mermaid)

Render with a Mermaid-compatible viewer or the VS Code Markdown Preview Enhanced extension.

```mermaid
flowchart TD
    subgraph INIT["① Construction  —  AdvectionParams(...)"]
        direction TB
        I1["Required inputs<br/>T_sc, T_sf, T_a<br/>ustar_f, ustar_c<br/>H_f, LE_f, avail_ratio, le_factor<br/>zom_f, zom_c, lm_option, lm_zshift<br/>Hmax, dz, dx, Lx"]
        I2{{"Moisture init<br/>__post_init__"}}
        I3A["RH_c / RH_f given<br/>→ convert to Q via<br/>Q_from_RH(T, RH)"]
        I3B["Q_c / Q_f given<br/>→ validate RH physical"]
        I3C["Neither given<br/>→ default RH<br/>(60 % crop, 10 % fallow)"]
        STORED["Stored state<br/>─────────────<br/>T_sc, T_sf, T_a<br/>Q_c, Q_f, Q_a  ← canonical<br/>ustar_f, ustar_c<br/>H_f, LE_f, avail_ratio, le_factor<br/>zom, SW_in<br/>─────────────<br/>Derived (cached_property)<br/>z, x, lm, h_c, d_c …<br/>Computed (@property)<br/>RH_c = f(T_sc, Q_c)<br/>RH_f = f(T_sf, Q_f)<br/>RNmG_up = H_f + LE_f<br/>RNmG_down = avail_ratio · RNmG_up<br/>LE_c = le_factor · RNmG_down<br/>H_c = (1 − le_factor) · RNmG_down"]
        I1 --> I2
        I2 --> I3A & I3B & I3C
        I3A & I3B & I3C --> STORED
    end

    subgraph RAD["② Radiation closure  —  solve_surface_radiation_inplace(fix=...)"]
        direction TB
        R1{{"fix = 'alpha_c'<br/>(most common)"}}
        R2{{"fix = 'alpha_f'"}}
        R1A["Solve symbolically<br/>for SW, alpha_f<br/>given alpha_c fixed<br/>→ writes SW_in, alpha_f"]
        R2A["Solve symbolically<br/>for SW, alpha_c<br/>given alpha_f fixed<br/>→ writes SW_in, alpha_c"]
        RNOTES["Uses: T_sf, T_sc, T_a<br/>H_f, LE_f, avail_ratio<br/>ε_f, ε_c, ε_a, G"]
        R1 --> R1A
        R2 --> R2A
        RNOTES -.->|reads| R1
        RNOTES -.->|reads| R2
    end

    subgraph BCS["③ Surface boundary conditions  —  two alternative methods"]
        direction TB
        B1{{"update_surface_BCs<br/>_from_reference()"}}
        B2{{"update_surface_BCs<br/>_from_zom()"}}
        B1N["Log-law reference height<br/>z_ref = z[0]  (grid bottom)<br/>ln(Hmax / z_ref)<br/>→ writes T_sc, T_sf, Q_c, Q_f"]
        B2N["Log-law reference at<br/>z0h = zom / 5  (heat roughness)<br/>ln(Hmax / z0h)<br/>→ writes T_sc, T_sf, Q_c, Q_f"]
        BCNOTE["Both use:<br/>H_c, H_f, LE_c, LE_f<br/>ustar_c, ustar_f, k<br/>rho·cp (H→T),  Lv (LE→Q)"]
        B1 --> B1N
        B2 --> B2N
        BCNOTE -.->|reads| B1
        BCNOTE -.->|reads| B2
    end

    subgraph REC["④ Heat reconciliation  —  reconcile_heat_with_prescribed_surface_T(mode=..., ref=...)"]
        direction TB
        M1{{"mode = 'update_ustar'"}}
        M2{{"mode = 'update_H'"}}
        M1A["Keep H_f, le_factor fixed<br/>Solve u*_f from H_f and dT_f<br/>Solve u*_c from H_c and dT_c<br/>→ writes ustar_f, ustar_c"]
        M2A["Keep ustar_f, ustar_c fixed<br/>Solve H_f from u*_f and dT_f<br/>Solve le_factor from H_c target<br/>→ writes H_f, le_factor"]
        REFNOTE["ref = 'z_ref'  →  ln(Hmax / z[0])<br/>ref = 'z0h'   →  ln(Hmax / zom/5)<br/>(must match BC method above)"]
        M1 --> M1A
        M2 --> M2A
        REFNOTE -.->|ref height| M1
        REFNOTE -.->|ref height| M2
    end

    subgraph SOLVER["⑤ Advection solver"]
        direction LR
        S1["uniform_T(p) / uniform_Q(p)<br/>─────────────<br/>Build upwind log-profile<br/>from T_sf, T_sc, T_a<br/>and Q_f, Q_c, Q_a<br/>→ initial column at x=0"]
        S2["integrate_T_step / integrate_H2O_step<br/>─────────────<br/>Implicit Thomas tridiagonal solve<br/>marching in x<br/>A = lm · u*  (eddy diffusivity)<br/>B = 1/U(z)  (advection)<br/>→ T(x,z), Q(x,z), FluxT, FluxQ"]
        S1 -->|column at x=0| S2
    end

    INIT -->|"p constructed"| RAD
    RAD -->|"SW_in, alpha updated"| BCS
    BCS -->|"T_sc, T_sf, Q_c, Q_f updated"| REC
    REC -->|"u* / H_f / le_factor consistent"| BCS2["update_surface_BCs (2nd call)<br/>re-derive T_s and Q_s<br/>from revised fluxes"]
    BCS2 --> SOLVER

    style INIT fill:#dbeafe,stroke:#3b82f6
    style RAD  fill:#fef3c7,stroke:#f59e0b
    style BCS  fill:#d1fae5,stroke:#10b981
    style REC  fill:#ede9fe,stroke:#7c3aed
    style SOLVER fill:#fee2e2,stroke:#ef4444
```

---

## Notes on BC variants

### Stage ③ — which method to use?

| Method | Reference height for log-law | When to use |
|--------|------------------------------|-------------|
| `update_surface_BCs_from_reference` | `z[0]` — first model grid level | Default; consistent with model grid |
| `update_surface_BCs_from_zom` | `zom / 5` — heat roughness sublayer | When you want surface T/Q closer to the aerodynamic roughness layer |

### Stage ④ — which mode to use?

| Mode | Keeps fixed | Adjusts | When to use |
|------|-------------|---------|-------------|
| `update_ustar` | `H_f`, `le_factor` | `ustar_f`, `ustar_c` | Prescribed fluxes, derive consistent friction velocity |
| `update_H` | `ustar_f`, `ustar_c` | `H_f`, `le_factor` | Prescribed wind profile, derive consistent heat fluxes |

> **Important:** the `ref=` argument to `reconcile_heat_with_prescribed_surface_T` must match the
> reference height used in Stage ③ (`ref='z_ref'` with `_from_reference`, `ref='z0h'` with `_from_zom`).

---

## Current constructor call (as of 2026-03-10)

```python
p = AdvectionParams(
    ...,
    lm_zshift=0.0005,
    zom_c=0.0005,
    avail_ratio=1.4,
    le_factor=1.4,
)
p.solve_surface_radiation_inplace(fix="alpha_c")
p.update_surface_BCs_from_reference(update_T=True, update_Q=True)   # ③ first pass
p.sync_surface_moisture_inplace("RH_from_Q")                        # inspect only
p.reconcile_heat_with_prescribed_surface_T(mode="update_ustar", ref="z_ref")  # ④
p.update_surface_BCs_from_reference(update_T=True, update_Q=True)   # ③ second pass
```